# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


## Lectura de la imagen

El grid es 5x6. Filas 0 a 4 de arriba hacia abajo, columnas 0 a 5 de izquierda a derecha.

|       | c0 | c1 | c2 | c3 | c4 | c5 |
|-------|----|----|----|----|----|----|
| **r0** | START | piso | piso | estanteria | piso | **ENTREGA +10** (T) |
| **r1** | piso | estanteria | resbaloso | piso | peligro -3 | piso |
| **r2** | piso | resbaloso | **CARGA +2** (T) | piso | estanteria | piso |
| **r3** | piso | piso | piso | resbaloso | piso | **MORTAL -10** (T) |
| **r4** | piso | peligro -3 | estanteria | piso | piso | piso |

- Estanterias (paredes): `(0,3)`, `(1,1)`, `(2,4)`, `(4,2)`
- Resbalosos: `(1,2)`, `(2,1)`, `(3,3)`
- Terminales: `(0,5)=+10`, `(2,2)=+2`, `(3,5)=-10`
- Peligros no terminales: `(1,4)=-3`, `(4,1)=-3`

Con 4 estanterias quedan **26 estados** de los 30 del grid.

Dos observaciones que importan para leer la politica despues:

1. La estanteria en `(0,3)` corta el pasillo de arriba, asi que desde el oeste no se puede llegar en linea recta a la entrega.
2. Para alcanzar `(0,5)` viniendo del oeste hay que pasar por `(1,4)`, que es el peligro `-3`. La unica alternativa es rodear por abajo, pero eso pasa pegado al `-10` de `(3,5)`. El `-3` es practicamente un peaje.


In [ ]:
import numpy as np


class WarehouseMDP:
    """
    Almacen 5x6 leido de la imagen del enunciado.

        state  = (row, col)
        action = (dr, dc)
        R(s)   = recompensa del estado actual
        T(s,a,s') = P(s' | s,a)

    Los parametros del constructor quedan expuestos para poder
    correr los experimentos de la Parte 5 sin reescribir la clase.
    """

    def __init__(self, living_reward=-1.0, gamma=0.9, p_slippery_intended=0.60):
        self.height = 5
        self.width = 6

        self.start = (0, 0)

        # Estanterias
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): +10.0,   # zona de entrega
            (2, 2): +2.0,    # estacion de carga
            (3, 5): -10.0,   # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = living_reward
        self.gamma = gamma

        # Piso normal
        self.p_intended = 0.90
        self.p_perpendicular = 0.05

        # Piso resbaloso
        self.p_slippery_intended = p_slippery_intended
        self.p_slippery_perpendicular = (1.0 - p_slippery_intended) / 2.0

        self.actions = [
            (-1, 0),  # UP
            (1, 0),   # DOWN
            (0, -1),  # LEFT
            (0, 1),   # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s) depende solo del estado actual.
        # El orden importa: un terminal nunca cobra living_reward.
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve [(next_state, probability), ...] para T(s,a,s').
        """
        # Terminal absorbente: una vez alli, el robot se queda.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Aqui esta la diferencia con el GridWorld de clase:
        # las probabilidades ya no son fijas, dependen del piso
        # en el que el robot esta parado AHORA.
        if state in self.slippery_states:
            p_go = self.p_slippery_intended
            p_side = self.p_slippery_perpendicular
        else:
            p_go = self.p_intended
            p_side = self.p_perpendicular

        # Perpendiculares a (dr, dc). Como ambas desviaciones
        # tienen la misma probabilidad, no hace falta distinguir
        # cual es la izquierda y cual la derecha.
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_go),
            (perp1, p_side),
            (perp2, p_side),
        ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (
                state[0] + next_action[0],
                state[1] + next_action[1],
            )
            # Si sale del grid o choca con una estanteria, se queda.
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))

        return transitions



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    # Sum_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # V_0(s) = 0 para todo s
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        # Copia para actualizacion sincronica: todo V_{k+1}
        # se calcula usando unicamente V_k.
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                # En un terminal no hay decision futura.
                V_new[state] = grid.get_reward(state)
            else:
                # max_a Sum_{s'} T(s,a,s') V_k(s')
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                # V_{k+1}(s) = R(s) + gamma * max_a Sum T V_k
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state]),
            )

        V = V_new

        # Criterio de parada: max_s |V_{k+1}(s) - V_k(s)| < theta
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a Sum_{s'} T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V),
        )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    """
    V_{k+1}^pi(s) = R(s) + gamma * Sum_{s'} T(s,pi(s),s') V_k^pi(s')

    Sin max: la politica ya eligio la accion.
    """
    V = {state: 0.0 for state in grid.states()}

    for sweep in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(
                        grid, state, policy[state], V
                    )
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state]),
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V, sweep + 1


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a Sum_{s'} T(s,a,s') V^pi(s')
    # Es exactamente extract_policy, pero aplicado a V^pi
    # en vez de a V*.
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. Politica inicial arbitraria: todos hacia UP.
    policy = {
        state: grid.actions[0]
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        # 2. Evaluacion
        V, sweeps = policy_evaluation(grid, policy, threshold)

        # 3. Mejora
        new_policy = policy_improvement(grid, V)

        changed = sum(
            1 for s in new_policy if new_policy[s] != policy[s]
        )

        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": sweeps,
            "changed_actions": changed,
        })

        policy = new_policy

        # 4. Estable: ninguna accion cambio.
        if changed == 0:
            break

    return policy, V, history



## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


---

## Parte 5 — Interpretacion

Para responder con datos y no de memoria, primero una utilidad que sigue la
politica desde `START` tomando siempre la accion intencionada, sin ruido.
No es una simulacion del MDP: es solo la ruta que el robot *intenta* recorrer.


In [ ]:
def rollout(grid, policy, max_steps=40):
    """Ruta intencionada desde START siguiendo la politica."""
    s = grid.start
    path = [s]
    for _ in range(max_steps):
        if grid.is_terminal(s):
            break
        a = policy[s]
        nxt = (s[0] + a[0], s[1] + a[1])
        if not grid.is_valid_state(nxt):
            nxt = s
        if nxt == s:
            break
        s = nxt
        path.append(s)
    return path


NOMBRES = {(0, 5): "ENTREGA +10", (2, 2): "CARGA +2", (3, 5): "MORTAL -10"}


def destino(grid, policy):
    end = rollout(grid, policy)[-1]
    return NOMBRES.get(end, f"no termina (queda en {end})")


def resolver(nombre, grid, mostrar=True):
    V, n = value_iteration(grid)
    pi = extract_policy(grid, V)
    if mostrar:
        print("=" * 60)
        print(nombre)
        print("=" * 60)
        print(f"Iteraciones: {n}    V(START) = {V[grid.start]:+.3f}")
        print()
        print_policy(grid, pi)
        print()
        print("Ruta:", " -> ".join(str(x) for x in rollout(grid, pi)))
        print("Destino:", destino(grid, pi))
        print()
    return V, pi


V_base, pi_base = resolver("BASE   living=-1.0   gamma=0.9   slip=0.60", WarehouseMDP())


### Respuestas

**1. Desde `START`, el robot va a la carga `+2`, no a la entrega `+10`.**

La ruta es `(0,0) → (0,1) → (0,2) → (1,2) → (2,2)`: cuatro pasos hasta el `+2`.
Llegar a la entrega costaria siete pasos y ademas obliga a cruzar el peligro `-3`
de `(1,4)`. Con `living_reward = -1` esos pasos de mas se comen la diferencia
entre las dos recompensas.

**2. Por que una recompensa menor puede ser optima.**

Porque lo que se maximiza no es la recompensa terminal sino la suma descontada
de todo el trayecto. Con `gamma = 0.9`, un `+10` a siete pasos vale
`0.9^7 * 10 = 4.78` antes de descontar el costo del camino, mientras que un `+2`
a cuatro pasos vale `0.9^4 * 2 = 1.31` pero con tres pasos menos de costo y sin
peaje de `-3`. Cerca, y el `+2` gana. `V(START) = -2.575` es negativo: incluso el
mejor plan disponible pierde plata, y el robot elige la perdida mas chica.

**3. Donde el piso resbaloso cambia la decision.**

El estado bisagra es `(1,2)`, el resbaloso que esta justo encima de la carga.
Ahi se decide bajar al `+2` o seguir a la derecha hacia el `+10`, y es el unico
estado que cambia de accion en el Experimento C. Los otros dos resbalosos pesan
distinto: `(2,1)` esta en una zona que el plan optimo ya evita, y `(3,3)` esta
en el corredor de abajo, cerca del `-10`, donde el resbalon es peligroso de
verdad. Por eso toda esa esquina apunta hacia arriba, alejandose.

**4. El papel del costo por paso `-1`.**

Es lo que le pone precio al tiempo. Sin el, el robot podria dar vueltas gratis
hasta encontrar el `+10`. Con `-1` cada paso duele, y por eso prefiere el premio
cercano y modesto. El Experimento A y el Bonus muestran que es justamente este
parametro, y no `gamma`, el que decide entre `+2` y `+10`.

**5. Por que `T(s,a,s')` ya no puede tener las mismas probabilidades para todos
los estados.**

Porque el ruido es una propiedad del piso, no de la accion. En el GridWorld de
clase todo el mundo era igual de resbaloso (`0.8 / 0.1 / 0.1`), asi que las
probabilidades se podian escribir como constantes de la clase. Aca, pararse en
`(1,2)` no es lo mismo que pararse en `(1,3)`: el reparto es `0.60 / 0.20 / 0.20`
en uno y `0.90 / 0.05 / 0.05` en el otro. La funcion tiene que mirar `state`
antes de repartir la probabilidad. Formalmente `T` siempre dependio de `s`, lo
que cambia es que ahora esa dependencia se nota.


### Experimento A — Menos costo por paso (`living_reward = -0.1`)

**Prediccion antes de correr:** si caminar deja de doler, el `+10` deberia
imponerse sobre el `+2`, porque la distancia extra ya casi no cuesta.


In [ ]:
V_a, pi_a = resolver("EXP A   living=-0.1", WarehouseMDP(living_reward=-0.1))

cambios = [(s, ARROWS[pi_base[s]], ARROWS[pi_a[s]])
           for s in pi_a if pi_base[s] != pi_a[s]]
print("Estados que cambian de accion respecto a la base:", len(cambios))
for s, antes, ahora in cambios:
    print(f"   {s}: {antes} -> {ahora}")

Se cumple. `V(START)` pasa de `-2.575` a `+1.649`, o sea que el plan ahora es
rentable, y la ruta se va hasta la entrega. Solo cambian tres estados, pero uno
de ellos es `(1,2)`, y con eso alcanza para redirigir todo el trayecto.

Detalle que vale la pena mirar: la ruta pasa por `(1,4)`, el peligro `-3`.
Con el costo por paso tan bajo, pagar el `-3` una vez sale mas barato que rodear
por abajo bordeando el `-10`.


### Experimento B — Piso muy resbaloso (`0.60 → 0.40`)

**Prediccion:** el robot deberia esquivar las celdas resbalosas, o al menos
bajarles el valor lo suficiente como para buscar otro camino.


In [ ]:
V_b, pi_b = resolver("EXP B   slip 0.60 -> 0.40", WarehouseMDP(p_slippery_intended=0.40))

cambios = [s for s in pi_b if pi_base[s] != pi_b[s]]
print("Estados que cambian de accion:", len(cambios))
print()
print("Valor de los estados resbalosos:")
for s in sorted(WarehouseMDP().slippery_states):
    print(f"   {s}: {V_base[s]:+7.3f}  ->  {V_b[s]:+7.3f}")

La prediccion falla, y el porque es lo interesante: **ningun estado cambia de
accion**. Los valores si empeoran, sobre todo `(2,1)` que cae de `-0.063` a
`-0.670`, pero la politica es identica.

La razon es que `(1,2)` es un cuello de botella. Con `(1,1)` y `(0,3)` bloqueados,
no existe una ruta alternativa a la carga que evite ese resbaloso, asi que el
robot lo cruza igual, solo que ahora le cuesta mas caro. Mas ruido encarece el
plan sin cambiarlo cuando no hay plan B.

`V(START)` baja apenas, de `-2.575` a `-2.706`.


### Experimento C — Mas paciencia (`gamma = 0.99`)

**Prediccion:** con menos descuento, el `+10` lejano pesa mas y deberia ganarle
al `+2` cercano.


In [ ]:
V_c, pi_c = resolver("EXP C   gamma=0.99", WarehouseMDP(gamma=0.99))

cambios = [(s, ARROWS[pi_base[s]], ARROWS[pi_c[s]])
           for s in pi_c if pi_base[s] != pi_c[s]]
print("Estados que cambian de accion:", len(cambios))
for s, antes, ahora in cambios:
    print(f"   {s}: {antes} -> {ahora}")

Se cumple, y de la forma mas economica posible: **cambia un solo estado**,
`(1,2)`, de bajar a seguir derecho. Ese unico cambio reorienta la ruta completa
hacia la entrega.

Ahi se ve que `(1,2)` es donde se juega la decision del problema. Todo lo demas
de la politica es reaccion a esa eleccion.


### Bonus — Donde esta el punto de quiebre de `living_reward`

Barrido de `living_reward` mirando a que terminal apunta la ruta desde `START`.


In [ ]:
print("Barrido grueso:")
anterior = None
for i in range(41):
    lr = round(-2.0 + 0.05 * i, 2)
    g = WarehouseMDP(living_reward=lr)
    V, _ = value_iteration(g)
    d = destino(g, extract_policy(g, V))
    if d != anterior:
        print(f"   living_reward = {lr:+.2f}  ->  {d}")
        anterior = d

print()
print("Refinamiento:")
anterior = None
for i in range(41):
    lr = round(-0.82 + 0.001 * i, 4)
    g = WarehouseMDP(living_reward=lr)
    V, _ = value_iteration(g)
    d = destino(g, extract_policy(g, V))
    if d != anterior:
        print(f"   living_reward = {lr:+.4f}  ->  {d}")
        anterior = d

**El quiebre esta en `living_reward ≈ -0.80`**, entre `-0.799` y `-0.798`.

- Con un costo por paso **mas caro que `-0.80`** el robot se conforma con la carga `+2`.
- Con un costo **mas barato que `-0.80`** se anima a cruzar el almacen hasta la entrega `+10`.

Es un cambio abrupto, no gradual: la politica optima es una funcion escalonada de
los parametros. Un ajuste minusculo del costo por paso, de `-0.799` a `-0.798`,
da vuelta el comportamiento entero del robot. Ese es el punto de fondo del lab:
la recompensa no describe *que hacer*, describe *que preferir*, y traducir eso a
conducta puede ser mucho mas sensible de lo que uno espera al escribir los numeros.
